# Imports

In [1]:
import torch
from torch.autograd import grad as autograd
from torch.distributions import Normal
import numpy as np

# Create Example Data, assuming a Normal Distribution

In [182]:
torch.manual_seed(123)
n = 10
transf_function = torch.exp

y = torch.randn(n).reshape(-1,1)

mu = torch.randn(n).reshape(-1,1)
mu.requires_grad = True

sigma = torch.randn(n).reshape(-1,1)
sigma.requires_grad = True
sigma_exp = transf_function(sigma)

params = [mu, sigma]

# Automatic Derivation

In [222]:
normal_dist = Normal(loc=mu, scale=sigma_exp)
nll = torch.nansum(normal_dist.log_prob(y))
grad = autograd(nll, inputs=params, create_graph=True)
hess = [autograd(grad[i].nansum(), inputs=params[i], retain_graph=True)[0] for i in range(len(grad))]

grad_auto = torch.cat(grad,dim=1).detach().numpy().round(4)
hess_auto = torch.cat(hess,dim=1).detach().numpy().round(4)

print(f"Automatic Gradients: \n {grad_auto}\n\n Automatic Hessians: \n {hess_auto}")

Automatic Gradients: 
 [[ -0.3406  -0.8904]
 [  0.0297  -0.9848]
 [ -1.3153  -0.2048]
 [ -5.3741   3.8672]
 [-10.5738  15.3867]
 [ -1.7292   0.3203]
 [ -0.1085  -0.8987]
 [  0.2926  -0.9617]
 [  3.4165   1.7154]
 [  0.4062  -0.8707]]

 Automatic Hessians: 
 [[-1.05830e+00 -2.19200e-01]
 [-5.82000e-02 -3.04000e-02]
 [-2.17540e+00 -1.59050e+00]
 [-5.93380e+00 -9.73450e+00]
 [-6.82290e+00 -3.27735e+01]
 [-2.26470e+00 -2.64060e+00]
 [-1.16200e-01 -2.02600e-01]
 [-2.23240e+00 -7.67000e-02]
 [-4.29860e+00 -5.43090e+00]
 [-1.27600e+00 -2.58700e-01]]


# Analytical Derivation

In [223]:
grad_mu = (y - mu) / (transf_function(sigma) ** 2)
grad_sigma = -1 + ((y - mu) ** 2) / (transf_function(sigma) ** 2)
grad_analytical = torch.cat([grad_mu, grad_sigma],axis=1).reshape(-1,2).detach().numpy().round(4)

hess_mu = -1 / (transf_function(sigma) ** 2)
hess_sigma = -2*((y - mu) ** 2) / (transf_function(sigma) ** 2)
hess_analytical = torch.cat([hess_mu, hess_sigma],axis=1).reshape(-1,2).detach().numpy().round(4)

print(f"Analytical Gradients: \n {grad_analytical} \n\n Analytical Hessians: \n {hess_analytical}")

Analytical Gradients: 
 [[ -0.3406  -0.8904]
 [  0.0297  -0.9848]
 [ -1.3153  -0.2048]
 [ -5.3741   3.8672]
 [-10.5738  15.3867]
 [ -1.7292   0.3203]
 [ -0.1085  -0.8987]
 [  0.2926  -0.9617]
 [  3.4165   1.7154]
 [  0.4062  -0.8707]] 

 Analytical Hessians: 
 [[-1.05830e+00 -2.19200e-01]
 [-5.82000e-02 -3.04000e-02]
 [-2.17540e+00 -1.59050e+00]
 [-5.93380e+00 -9.73450e+00]
 [-6.82290e+00 -3.27735e+01]
 [-2.26470e+00 -2.64060e+00]
 [-1.16200e-01 -2.02600e-01]
 [-2.23240e+00 -7.67000e-02]
 [-4.29860e+00 -5.43090e+00]
 [-1.27600e+00 -2.58700e-01]]


# Compare

In [224]:
print(f"Analytical and Automatic Gradients are equal: {np.array_equal(grad_auto, grad_analytical)}")
print(f"Analytical and Automatic Hessians are equal: {np.array_equal(hess_auto, hess_analytical)}")

Analytical and Automatic Gradients are equal: True
Analytical and Automatic Hessians are equal: True


# Natural Gradient

# Natural Gradient

To derive the Fisher Information Matrix (FIM) and its inverse using the given partial derivatives for parameters $\mu_x$ and $\sigma_x$, we start by examining the second-order derivatives provided. We are focusing on a parameterization where $\sigma_x$ represents the logarithm of the standard deviation, which alters the scale and interpretation of its derivatives.

### Deriving the Fisher Information Matrix (FIM)

The FIM is defined for each parameter as the negative expectation of the second-order derivative of the log-likelihood. The elements of the FIM can be calculated directly from the expressions given:

#### Fisher Information for $\mu_x$
The second derivative with respect to $\mu_x$ is constant across observations (does not depend on $y$):
    $ \frac{\partial^2 \log f(\cdot)}{\partial (\mu_x)^2} = -\frac{1}{\big(\exp(\sigma_{x})\big)^{2}} $
Given that this derivative is negative, the Fisher information for $\mu_x$ is:
    $I(\mu_x) = \frac{1}{\big(\exp(\sigma_{x})\big)^{2}} $

#### Fisher Information for $\sigma_x$
The second derivative with respect to $\sigma_x$ depends on $(y-\mu_x)^2$. Assuming $y$ is normally distributed around $\mu_x$ with variance $\exp(\sigma_x)^2$, the expectation $E[(y-\mu_x)^2]$ equals $\exp(\sigma_x)^2$:
    $\frac{\partial^2 \log f(\cdot)}{\partial (\sigma_x)^2} = -\frac{2(y-\mu_x)^2} {\big(\exp(\sigma_{x})\big)^{2}} $
Hence, the expected value of this derivative becomes:
$ E\left[-\frac{2(y-\mu_x)^2}{\big(\exp(\sigma_{x})\big)^{2}}\right] = -\frac{2 \exp(\sigma_x)^2}{\big(\exp(\sigma_{x})\big)^{2}} = -2 $
$ I(\sigma_x) = 2 $

### Fisher Information Matrix
The Fisher Information Matrix $I(\theta)$ with no cross-derivatives (since they are not provided and typically assumed to be zero unless specified otherwise) is:
$
I(\theta) =
\begin{bmatrix}
\frac{1}{\big(\exp(\sigma_{x})\big)^{2}} & 0 \ \
    0 & 2
\end{bmatrix}
$

### Inverse Fisher Information Matrix
The inverse FIM is obtained by taking the reciprocal of each diagonal element:
$
I(\theta)^{-1} =
\begin{bmatrix}
\big(\exp(\sigma_{x})\big)^{2} & 0 \ \
    0 & \frac{1}{2}
\end{bmatrix}
$

This matrix gives us the variances of the maximum likelihood estimators for $\mu_x$ and $\sigma_x$, showing that the estimator of $\mu_x$ has a variance of $\big(\exp(\sigma_{x})\big)^{2}$ and the estimator of $\sigma_x$ has a variance of $0.5$. This representation is aligned with the modified scale due to the log transformation of $\sigma_x$.

## Analytical

In [233]:
I_mu_inv = torch.ones_like(sigma) * (transf_function(sigma * 2)) 
#I_sigma_inv = torch.ones_like(sigma) * 0.5 * (transf_function(sigma * 2)) / (mu - y) ** 2
I_sigma_inv = torch.ones_like(sigma) * 0.5


nat_grad_mu = grad_mu * (-I_mu_inv)
nat_grad_sigma = grad_sigma * (-I_sigma_inv) 
nat_grad_analytical = torch.cat([nat_grad_mu, nat_grad_sigma],axis=1).reshape(-1,2).detach().numpy().round(4)
nat_grad_analytical

array([[ 0.3218,  0.4452],
       [-0.5112,  0.4924],
       [ 0.6046,  0.1024],
       [ 0.9057, -1.9336],
       [ 1.5497, -7.6934],
       [ 0.7636, -0.1602],
       [ 0.9337,  0.4494],
       [-0.1311,  0.4808],
       [-0.7948, -0.8577],
       [-0.3184,  0.4353]], dtype=float32)

## Automatic

The natural gradient is the product of the inverse of the FIM and the conventional gradient. Since we're working with a diagonal approximation, the inverse of the FIM is the expectation of the reciprocal of the diagonal elements of a Hessian. The Hessian matrix $H(\theta)$ of the log-likelihood function $\ell\left(\mu, \sigma^{\prime}\right)$ is composed of these second derivatives:
$$
H(\theta)=\left(\begin{array}{cc}
\frac{\partial^2 \ell}{\partial \mu^2} & \frac{\partial^2 \ell}{\partial \mu \partial \sigma^{\prime}} \\
\frac{\partial^2 \ell}{\partial \mu \partial \sigma^{\prime}} & \frac{\partial^2 \ell}{\partial \sigma^{\prime 2}}
\end{array}\right)
$$

Substituting the values we found:
$$
H(\theta)=\left(\begin{array}{cc}
-\sum_{i=1}^n \frac{1}{e^{2 \sigma^{\prime}}} & \sum_{i=1}^n\left(-2 \cdot \frac{\left(y_i-\mu\right)}{e^{2 \sigma^{\prime}}}\right) \\
\sum_{i=1}^n\left(-2 \cdot \frac{\left(y_i-\mu\right)}{e^{2 \sigma^{\prime}}}\right) & \sum_{i=1}^n\left(-2 \cdot \frac{\left(y_i-\mu\right)^2}{e^{2 \sigma^{\prime}}}\right)
\end{array}\right)
$$

Simplifying:
$$
H(\theta)=\left(\begin{array}{cc}
-\frac{n}{e^{2 \sigma^{\prime}}} & -2 \sum_{i=1}^n \frac{\left(y_i-\mu\right)}{e^{2 \sigma^{\prime}}} \\
-2 \sum_{i=1}^n \frac{\left(y_i-\mu\right)}{e^{2 \sigma^{\prime}}} & -2 \sum_{i=1}^n \frac{\left(y_i-\mu\right)^2}{e^{2 \sigma^{\prime}}}
\end{array}\right)
$$

Thus, the Hessian matrix for the log-likelihood function $\ell\left(\mu, \sigma^{\prime}\right)$ is:
$$
H(\theta)=\left(\begin{array}{cc}
-\frac{n}{e^{2 \sigma^{\prime}}} & -2 \sum_{i=1}^n \frac{\left(y_i-\mu\right)}{e^{2 \sigma^{\prime}}} \\
-2 \sum_{i=1}^n \frac{\left(y_i-\mu\right)}{e^{2 \sigma^{\prime}}} & -2 \sum_{i=1}^n \frac{\left(y_i-\mu\right)^2}{e^{2 \sigma^{\prime}}}
\end{array}\right)
$$

Once we compute the negative expected value of the Hessian, the expression simplifies to the analytical formulation of the FIM

$$
I(\theta) = - \mathbb{E} \left[ H(\theta) \right]
$$

$$
I(\theta)^{-1} =
\begin{bmatrix}
\big(\exp(\sigma_{x})\big)^{2} & 0 \ \
    0 & \frac{1}{2}
\end{bmatrix}
$$

Since $y_i$ are i.i.d from a Normal Distribution $\mathcal{N}(\mu, e^{2*\sigma_x})$

$$
\mathbb{E} \left[ (y_i - \mu)^2 \right] = e^{2\sigma_x}
$$

In [238]:
modified_hess = hess.copy()
fim_diag_2 =   torch.ones(n,1) * 2
modified_hess[1] =  - torch.tensor(fim_diag_2)
modified_hess

/var/folders/8b/4kssy4kj57jb6rv1pmm5w8h80000gn/T/ipykernel_19989/383496126.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  modified_hess[1] =  - torch.tensor(fim_diag_2)


[tensor([[-1.0583],
         [-0.0582],
         [-2.1754],
         [-5.9338],
         [-6.8229],
         [-2.2647],
         [-0.1162],
         [-2.2324],
         [-4.2986],
         [-1.2760]]),
 tensor([[-2.],
         [-2.],
         [-2.],
         [-2.],
         [-2.],
         [-2.],
         [-2.],
         [-2.],
         [-2.],
         [-2.]])]

In [239]:
nat_grad_auto = torch.cat([grad[i] / modified_hess[i] for i in range(len(grad))], axis=1).detach().numpy().round(4)
nat_grad_auto

array([[ 0.3218,  0.4452],
       [-0.5112,  0.4924],
       [ 0.6046,  0.1024],
       [ 0.9057, -1.9336],
       [ 1.5497, -7.6934],
       [ 0.7636, -0.1602],
       [ 0.9337,  0.4494],
       [-0.1311,  0.4808],
       [-0.7948, -0.8577],
       [-0.3184,  0.4353]], dtype=float32)